# HW1 - CUDA Matrix Multiplication

**Author:** sivasurya.chandran@sjsu.edu

**Personal Parameters** (see Section 0.1 of the standing requirements):

| SID4 | SEED | SLICE | HP_ID | CLS_A | CLS_B |
|------|------|-------|-------|-------|-------|
| 3215 | 3215 | 215   | 5     | 5     | 2     |

HP_ID = 5 → **Schedule-long** arm → hidden layers `[64, 32]`, learning rate `0.001`, **60 epochs** (vs. the 30-epoch baseline).

CLS_A/CLS_B are derived for completeness but are not referenced by any HW1 task (no per-class subsetting is required in this assignment), so they are not used further below.

> The PyTorch/TensorFlow experiments are in `neural_networks.ipynb`. This notebook is the CUDA section only.


## 0. Colab setup


In [1]:
import os, sys, glob

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Find the folder containing diabetes.csv anywhere under MyDrive (first match wins).
    hits = glob.glob('/content/drive/MyDrive/**/diabetes.csv', recursive=True)
    if hits:
        PROJECT_DIR = os.path.dirname(hits[0])
    else:
        # Fall back to the Colab upload widget if the file is not in Drive.
        print('diabetes.csv not found in Drive - upload diabetes.csv and matmul.cu now:')
        from google.colab import files
        files.upload()
        PROJECT_DIR = '/content'
else:
    PROJECT_DIR = os.getcwd()

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Files present:', sorted(os.listdir('.')))


Mounted at /content/drive
Working directory: /content/drive/MyDrive/DATA266/data266-3215
Files present: ['.git', '.gitignore', 'AI_USE.md', 'HW1_writeup.docx', 'METRICS.md', 'README.md', 'RUN_LOG.txt', 'cuda.ipynb', 'diabetes.csv', 'figures', 'matmul.cu', 'neural_networks.ipynb']


In [2]:
SID4 = 3215
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print(f"SID4={SID4}  SEED={SEED}  SLICE={SLICE}  HP_ID={HP_ID}  CLS_A={CLS_A}  CLS_B={CLS_B}")

import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)


SID4=3215  SEED=3215  SLICE=215  HP_ID=5  CLS_A=5  CLS_B=2


## CUDA Matrix Multiplication (5 points)

**Run this section on a GPU runtime:** `Runtime` -> `Change runtime type` -> `Hardware accelerator: T4 GPU` (or any available NVIDIA GPU), then `Runtime` -> `Run all`. The PyTorch/TensorFlow sections live in `neural_networks.ipynb` and are CPU-only; this notebook needs the GPU runtime.


### 1. Confirm GPU + CUDA toolkit are available

In [16]:
!nvidia-smi


Tue Sep  1 05:54:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             26W /   70W |     381MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


### 2. Upload / write `matmul.cu`

If you are running this notebook standalone on Colab, upload `matmul.cu` (from this submission)
into the Colab file browser, or run the cell below to write it directly from this notebook so the
`.cu` source is self-contained here as well.


In [18]:
%%writefile matmul.cu
// matmul.cu
// HW1 CUDA task: tiled matrix multiplication with CUDA C, benchmarked against a CPU baseline.
//
// SID4=3215 SEED=3215 (personal parameters; not used numerically by this kernel, recorded per
// the standing assignment requirements).
//
// Usage:
//   nvcc -O3 -arch=sm_75 matmul.cu -o matmul
//   ./matmul <N>          // N = square matrix dimension, e.g. 256, 1024, 4096
//
// Design notes on blocks/threads (see inline comments in matMulTiled for the full explanation):
//   - Each CUDA thread computes exactly one output element C[row][col].
//   - Threads are grouped into 2D thread blocks of size TILE x TILE (16x16 = 256 threads/block).
//   - The grid is a 2D array of blocks sized so that gridDim * blockDim covers the whole N x N
//     output matrix, i.e. grid = ceil(N/TILE) x ceil(N/TILE) blocks.
//   - Each block cooperatively loads TILE x TILE tiles of A and B into fast on-chip shared memory,
//     reused by all TILE*TILE threads in the block before advancing to the next tile along K.
//     This tiling cuts global-memory traffic by a factor of ~TILE compared to a naive kernel where
//     every thread re-reads full rows/columns of A and B from global memory.

#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <chrono>
#include <cuda_runtime.h>

#define TILE 16

#define CUDA_CHECK(call)                                                        \
    do {                                                                        \
        cudaError_t err = (call);                                               \
        if (err != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error %s:%d: %s\n", __FILE__, __LINE__,       \
                    cudaGetErrorString(err));                                   \
            exit(EXIT_FAILURE);                                                 \
        }                                                                        \
    } while (0)

// -----------------------------------------------------------------------------------------------
// Tiled GPU matrix multiplication kernel: C = A * B, all matrices N x N, row-major.
//
// Blocks and threads:
//   - blockDim = (TILE, TILE): a 2D block of TILE*TILE = 256 threads. Thread (tx, ty) within a
//     block is responsible for one element of the TILE x TILE output tile owned by this block.
//   - gridDim = (ceil(N/TILE), ceil(N/TILE)): enough 2D blocks to cover every TILE x TILE tile of
//     the N x N output matrix C. blockIdx.x/blockIdx.y select which output tile this block owns.
//   - Global row/col for this thread: row = blockIdx.y*TILE + threadIdx.y,
//                                      col = blockIdx.x*TILE + threadIdx.x.
//   - Parallelism: all N*N output elements are computed concurrently by different threads (subject
//     to how many can be resident on the SMs at once); within a block, threads cooperate via
//     __shared__ memory tiles As/Bs, loading one TILE x TILE tile of A and one of B per iteration
//     of the outer loop over the K dimension, syncing with __syncthreads() so every thread's
//     shared-memory reads happen after all loads finish (and before the next iteration overwrites
//     the tile).
// -----------------------------------------------------------------------------------------------
__global__ void matMulTiled(const float* __restrict__ A,
                             const float* __restrict__ B,
                             float* __restrict__ C,
                             int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;   // output row this thread computes
    int col = blockIdx.x * TILE + tx;   // output column this thread computes

    float acc = 0.0f;
    int numTiles = (N + TILE - 1) / TILE;

    for (int t = 0; t < numTiles; ++t) {
        int aCol = t * TILE + tx;
        int bRow = t * TILE + ty;

        As[ty][tx] = (row < N && aCol < N) ? A[row * N + aCol] : 0.0f;
        Bs[ty][tx] = (bRow < N && col < N) ? B[bRow * N + col] : 0.0f;

        __syncthreads();  // wait until the whole tile is loaded before reading it

        #pragma unroll
        for (int k = 0; k < TILE; ++k) {
            acc += As[ty][k] * Bs[k][tx];
        }

        __syncthreads();  // wait until all threads finish reading before next tile overwrites it
    }

    if (row < N && col < N) {
        C[row * N + col] = acc;
    }
}

// -----------------------------------------------------------------------------------------------
// CPU baseline: naive triple-loop matmul (single-threaded), used as the timing reference.
// -----------------------------------------------------------------------------------------------
void matMulCPU(const float* A, const float* B, float* C, int N) {
    for (int i = 0; i < N; ++i) {
        for (int j = 0; j < N; ++j) {
            float sum = 0.0f;
            for (int k = 0; k < N; ++k) {
                sum += A[i * N + k] * B[k * N + j];
            }
            C[i * N + j] = sum;
        }
    }
}

void fillRandom(float* mat, int N, unsigned seed) {
    srand(seed);
    for (int i = 0; i < N * N; ++i) {
        mat[i] = static_cast<float>(rand()) / RAND_MAX;
    }
}

double maxAbsDiff(const float* a, const float* b, int N) {
    double maxDiff = 0.0;
    for (int i = 0; i < N * N; ++i) {
        double d = fabs(static_cast<double>(a[i]) - static_cast<double>(b[i]));
        if (d > maxDiff) maxDiff = d;
    }
    return maxDiff;
}

int main(int argc, char** argv) {
    const unsigned SEED = 3215;  // SID4, for reproducible random inputs
    int N = 1024;
    if (argc > 1) N = atoi(argv[1]);

    printf("Matrix size: %d x %d (TILE=%d)\n", N, N, TILE);

    size_t bytes = static_cast<size_t>(N) * N * sizeof(float);
    float* h_A = static_cast<float*>(malloc(bytes));
    float* h_B = static_cast<float*>(malloc(bytes));
    float* h_C_gpu = static_cast<float*>(malloc(bytes));
    float* h_C_cpu = static_cast<float*>(malloc(bytes));

    fillRandom(h_A, N, SEED);
    fillRandom(h_B, N, SEED + 1);

    // ---------------- CPU baseline timing ----------------
    // Skip the full O(N^3) CPU run for N=4096 (would take a very long time single-threaded);
    // still verify correctness on smaller sizes.
    double cpuMs = -1.0;
    bool ranCpu = (N <= 2048);
    if (ranCpu) {
        auto cpuStart = std::chrono::high_resolution_clock::now();
        matMulCPU(h_A, h_B, h_C_cpu, N);
        auto cpuEnd = std::chrono::high_resolution_clock::now();
        cpuMs = std::chrono::duration<double, std::milli>(cpuEnd - cpuStart).count();
        printf("CPU time:            %10.3f ms\n", cpuMs);
    } else {
        printf("CPU time:            skipped (N=%d too large for naive single-thread CPU loop)\n", N);
    }

    // ---------------- GPU setup ----------------
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));

    cudaEvent_t evH2DStart, evH2DEnd, evKernelStart, evKernelEnd, evD2HStart, evD2HEnd;
    CUDA_CHECK(cudaEventCreate(&evH2DStart));
    CUDA_CHECK(cudaEventCreate(&evH2DEnd));
    CUDA_CHECK(cudaEventCreate(&evKernelStart));
    CUDA_CHECK(cudaEventCreate(&evKernelEnd));
    CUDA_CHECK(cudaEventCreate(&evD2HStart));
    CUDA_CHECK(cudaEventCreate(&evD2HEnd));

    // ---------------- H2D transfer ----------------
    CUDA_CHECK(cudaEventRecord(evH2DStart));
    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(evH2DEnd));

    // ---------------- Kernel launch ----------------
    dim3 blockDim(TILE, TILE);
    dim3 gridDim((N + TILE - 1) / TILE, (N + TILE - 1) / TILE);
    printf("Grid: (%d, %d) blocks, Block: (%d, %d) threads => %d threads total\n",
           gridDim.x, gridDim.y, blockDim.x, blockDim.y,
           gridDim.x * gridDim.y * blockDim.x * blockDim.y);

    // Warm-up launch (excluded from timing) so the timed run isn't skewed by first-launch overhead.
    matMulTiled<<<gridDim, blockDim>>>(d_A, d_B, d_C, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaEventRecord(evKernelStart));
    matMulTiled<<<gridDim, blockDim>>>(d_A, d_B, d_C, N);
    CUDA_CHECK(cudaEventRecord(evKernelEnd));
    CUDA_CHECK(cudaEventSynchronize(evKernelEnd));
    CUDA_CHECK(cudaGetLastError());

    // ---------------- D2H transfer ----------------
    CUDA_CHECK(cudaEventRecord(evD2HStart));
    CUDA_CHECK(cudaMemcpy(h_C_gpu, d_C, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(evD2HEnd));
    CUDA_CHECK(cudaEventSynchronize(evD2HEnd));

    float h2dMs = 0, kernelMs = 0, d2hMs = 0;
    CUDA_CHECK(cudaEventElapsedTime(&h2dMs, evH2DStart, evH2DEnd));
    CUDA_CHECK(cudaEventElapsedTime(&kernelMs, evKernelStart, evKernelEnd));
    CUDA_CHECK(cudaEventElapsedTime(&d2hMs, evD2HStart, evD2HEnd));
    float transferMs = h2dMs + d2hMs;
    float endToEndMs = h2dMs + kernelMs + d2hMs;

    printf("GPU H2D transfer:    %10.3f ms\n", h2dMs);
    printf("GPU kernel time:     %10.3f ms\n", kernelMs);
    printf("GPU D2H transfer:    %10.3f ms\n", d2hMs);
    printf("GPU H2D+D2H total:   %10.3f ms\n", transferMs);
    printf("GPU end-to-end time: %10.3f ms\n", endToEndMs);

    if (ranCpu) {
        double speedupKernelOnly = cpuMs / kernelMs;
        double speedupEndToEnd = cpuMs / endToEndMs;
        printf("Speedup (CPU / GPU kernel only):   %8.2fx\n", speedupKernelOnly);
        printf("Speedup (CPU / GPU end-to-end):    %8.2fx\n", speedupEndToEnd);

        double diff = maxAbsDiff(h_C_gpu, h_C_cpu, N);
        printf("Max abs diff GPU vs CPU: %.6e %s\n", diff, (diff < 1e-2) ? "(OK)" : "(CHECK!)");
    }

    // ---------------- Cleanup ----------------
    CUDA_CHECK(cudaEventDestroy(evH2DStart));
    CUDA_CHECK(cudaEventDestroy(evH2DEnd));
    CUDA_CHECK(cudaEventDestroy(evKernelStart));
    CUDA_CHECK(cudaEventDestroy(evKernelEnd));
    CUDA_CHECK(cudaEventDestroy(evD2HStart));
    CUDA_CHECK(cudaEventDestroy(evD2HEnd));
    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaFree(d_C));
    free(h_A);
    free(h_B);
    free(h_C_gpu);
    free(h_C_cpu);

    return 0;
}


Overwriting matmul.cu


### 3. Build with `nvcc`

In [19]:
# Detect the GPU's compute capability so this builds on T4 (sm_75), A100 (sm_80), L4 (sm_89), etc.
import subprocess
cc = subprocess.run(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True).stdout.strip().splitlines()[0].replace('.', '')
print('Detected compute capability: sm_' + cc)

!nvcc -O3 -arch=sm_{cc} matmul.cu -o matmul
!ls -la matmul


Detected compute capability: sm_75
-rwx------ 1 root root 1006696 Sep  1 05:54 matmul


### 4. Run for matrix sizes 256, 1024, 4096

The program times: CPU baseline (naive triple loop, skipped for N=4096 since a single-threaded
O(N^3) CPU pass at that size is impractically slow), GPU H2D+D2H transfer time, and GPU kernel time
(both measured with `cudaEvent` timers), then reports end-to-end GPU time and speedup.


In [20]:
!./matmul 256


Matrix size: 256 x 256 (TILE=16)
CPU time:                23.236 ms
Grid: (16, 16) blocks, Block: (16, 16) threads => 65536 threads total
GPU H2D transfer:         1.209 ms
GPU kernel time:          0.113 ms
GPU D2H transfer:         0.264 ms
GPU H2D+D2H total:        1.473 ms
GPU end-to-end time:      1.586 ms
Speedup (CPU / GPU kernel only):     206.28x
Speedup (CPU / GPU end-to-end):       14.65x
Max abs diff GPU vs CPU: 2.288818e-05 (OK)


In [21]:
!./matmul 1024


Matrix size: 1024 x 1024 (TILE=16)
CPU time:              3663.505 ms
Grid: (64, 64) blocks, Block: (16, 16) threads => 1048576 threads total
GPU H2D transfer:         2.143 ms
GPU kernel time:          5.808 ms
GPU D2H transfer:         2.891 ms
GPU H2D+D2H total:        5.034 ms
GPU end-to-end time:     10.842 ms
Speedup (CPU / GPU kernel only):     630.81x
Speedup (CPU / GPU end-to-end):      337.90x
Max abs diff GPU vs CPU: 9.155273e-05 (OK)


In [22]:
!./matmul 4096


Matrix size: 4096 x 4096 (TILE=16)
CPU time:            skipped (N=4096 too large for naive single-thread CPU loop)
Grid: (256, 256) blocks, Block: (16, 16) threads => 16777216 threads total
GPU H2D transfer:        28.992 ms
GPU kernel time:        196.829 ms
GPU D2H transfer:        47.662 ms
GPU H2D+D2H total:       76.655 ms
GPU end-to-end time:    273.484 ms


### 5. Profile with Nsight Systems (`nsys`)

Colab images ship `nsys` (Nsight Systems CLI). We profile the N=1024 case (representative size) and
export a summary that separates kernel time from memcpy (H2D/D2H) time.


In [23]:
!which nsys && nsys --version || echo 'nsys not available in this image'


nsys not available in this image


In [24]:
!nsys profile -o matmul_1024_profile --stats=true ./matmul 1024 || echo 'nsys profiling unavailable - use the ncu/nvprof cells below'


/bin/bash: line 1: nsys: command not found
nsys profiling unavailable - use the ncu/nvprof cells below


The `--stats=true` flag above prints a summary table directly to stdout, breaking down time spent
in `cudaMemcpy` (H2D/D2H) vs. the `matMulTiled` kernel vs. other CUDA API calls (`cudaMalloc`,
`cudaEventSynchronize`, etc.) - copy the "CUDA GPU Kernel Summary" and "CUDA GPU MemOps Summary"
sections from the output above into `METRICS.md` / the write-up as the required profiler output
that separates kernel time from transfer time.

If `nsys` is unavailable in your Colab image, fall back to legacy `nvprof` (older CUDA runtimes) or
Nsight Compute (`ncu`) instead - try the cells below and use whichever succeeds; report in the
write-up which profiler you actually used.


In [25]:
# Fallback profiler (legacy CUDA toolkits only). Safe to fail on newer images.
!nvprof ./matmul 1024 || echo 'nvprof not available (expected on CUDA 11+)'


Matrix size: 1024 x 1024 (TILE=16)
CPU time:              3306.403 ms
==5937== NVPROF is profiling process 5937, command: ./matmul 1024
Grid: (64, 64) blocks, Block: (16, 16) threads => 1048576 threads total
GPU H2D transfer:         2.041 ms
GPU kernel time:          5.842 ms
GPU D2H transfer:         2.911 ms
GPU H2D+D2H total:        4.951 ms
GPU end-to-end time:     10.794 ms
Speedup (CPU / GPU kernel only):     565.93x
Speedup (CPU / GPU end-to-end):      306.32x
Max abs diff GPU vs CPU: 9.155273e-05 (OK)
==5937== Profiling application: ./matmul 1024
==5937== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   78.80%  11.586ms         2  5.7929ms  5.7892ms  5.7965ms  matMulTiled(float const *, float const *, float*, int)
                   10.71%  1.5741ms         1  1.5741ms  1.5741ms  1.5741ms  [CUDA memcpy DtoH]
                   10.50%  1.5432ms         2  771.61us  740.21us  803.02us  [CUDA memcpy HtoD]
     

In [26]:
# Nsight Compute: kernel-level metrics (occupancy, memory throughput).
!ncu --set basic -o matmul_1024_ncu ./matmul 1024 || echo 'ncu not available in this image'


Matrix size: 1024 x 1024 (TILE=16)
CPU time:              3708.261 ms
==PROF== Connected to process 5995 (/content/drive/MyDrive/DATA266/data266-3215/matmul)
Grid: (64, 64) blocks, Block: (16, 16) threads => 1048576 threads total
==PROF== Profiling "matMulTiled" - 0: 0%....50%....100% - 9 passes
==PROF== Profiling "matMulTiled" - 1: 0%....50%....100% - 9 passes
GPU H2D transfer:         2.886 ms
GPU kernel time:       1714.722 ms
GPU D2H transfer:         3.886 ms
GPU H2D+D2H total:        6.772 ms
GPU end-to-end time:   1721.494 ms
Speedup (CPU / GPU kernel only):       2.16x
Speedup (CPU / GPU end-to-end):        2.15x
Max abs diff GPU vs CPU: 9.155273e-05 (OK)
==PROF== Disconnected from process 5995
==PROF== Report: /content/drive/MyDrive/DATA266/data266-3215/matmul_1024_ncu.ncu-rep


### 6. Collect results into the timing table

Paste the printed CPU/GPU timings from step 4 into the table below (also copy the final numbers
into `METRICS.md`).

| Matrix size | CPU (ms) | GPU kernel (ms) | H2D+D2H (ms) | End-to-end GPU (ms) | Speedup (CPU / end-to-end) |
|---|---|---|---|---|---|
| 256  | *(fill in)* | *(fill in)* | *(fill in)* | *(fill in)* | *(fill in)* |
| 1024 | *(fill in)* | *(fill in)* | *(fill in)* | *(fill in)* | *(fill in)* |
| 4096 | skipped (CPU) | *(fill in)* | *(fill in)* | *(fill in)* | N/A (no CPU baseline) |

### 7. Crossover discussion

After filling in the table, answer in 3-4 sentences: at what matrix size does GPU end-to-end time
(including transfer) first beat the CPU baseline in your measurements, and why isn't the crossover
at size 0? (Hint: fixed per-launch overheads - context/kernel launch latency, `cudaMalloc`,
H2D/D2H transfer setup - are roughly constant regardless of N, so they dominate at small N where
the O(N^3) compute savings are too small to outweigh them; the GPU only wins once N^3 growth in
compute time outpaces those fixed overheads.)
